# Curvature-Coupled Dark Energy: Evolution of the ISW Source

This notebook reproduces **Fig. 13** of the Curvature-Coupled Dark Energy (CCDE) paper.

The figure shows the conformal-time derivative of the Weyl potential,

$\Phi_{\rm W}'=\frac{d}{d\eta}\left(\frac{\Phi+\Psi}{2}\right),$

for three representative Fourier modes. This quantity is the local source entering the
linear late Integrated Sachs--Wolfe (ISW) effect.


In [ ]:

import os
from os.path import exists

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, AutoMinorLocator


## 1. Load CCDE perturbation outputs

The scalar perturbation files are loaded for the same $(\sigma,\alpha)$ grid used throughout
the paper. The data container retains the original structure,

`data["perturbations"][sigma_key][alpha_key][k_index]`.

The files are read from the public CCDE data directory and use the renamed `CCDE_...`
filename convention.


In [ ]:

# -------------------------------------------------
# Configuration
# -------------------------------------------------
data_address = "./../DataGenerator/transfer_functions/"
load_data = True

alphas = ["0.1", "0.05", "0.001", "-0.05", "-0.1"]
sigmas = ["0.001", "0.3", "1.", "1.5"]

# CLASS perturbation-output indices and their physical wavenumbers.
k_indices = [0, 1, 2, 3, 4, 5, 6]
k_output_values = [0.0001, 0.005, 0.001, 0.05, 0.01, 0.5, 0.1]


# -------------------------------------------------
# Helpers
# -------------------------------------------------
def skey(s):
    return f"sigma={s}"


def akey(a):
    return f"alpha={a}"


# -------------------------------------------------
# Build alpha/sigma arrays with stable indexing
# -------------------------------------------------
alpha = np.array(sorted(set(alphas), key=float), dtype=object)
sigma = np.array(sorted(set(sigmas), key=float), dtype=object)

alpha_to_j = {a: j for j, a in enumerate(alpha)}
sigma_to_i = {s: i for i, s in enumerate(sigma)}

out_arr = np.empty((len(sigma), len(alpha)), dtype=object)


# -------------------------------------------------
# Main data container
# -------------------------------------------------
# data["perturbations"][sigma_key][alpha_key][k_index]
data = {}
data.setdefault("perturbations", {})


# -------------------------------------------------
# Load CCDE perturbation files
# -------------------------------------------------
total_loaded_files = 0
missing_files = []

if load_data:
    for a_str in alpha:
        for s_str in sigma:

            output_dir = f"sigma{s_str}_alpha{a_str}"
            base_path = os.path.join(data_address, "run_" + output_dir)

            i = sigma_to_i[s_str]
            j = alpha_to_j[a_str]
            out_arr[i][j] = output_dir

            data["perturbations"].setdefault(skey(s_str), {})
            data["perturbations"][skey(s_str)].setdefault(akey(a_str), {})

            print(f"\033[94mChecking {output_dir}\033[0m")

            for k_ind in k_indices:

                pert_file = os.path.join(
                    base_path,
                    f"CCDE_sigma{s_str}_alpha{a_str}_perturbations_k{k_ind}_s.dat"
                )

                if exists(pert_file):
                    data["perturbations"][skey(s_str)][akey(a_str)][k_ind] = np.loadtxt(
                        pert_file
                    )
                    total_loaded_files += 1
                else:
                    data["perturbations"][skey(s_str)][akey(a_str)][k_ind] = None
                    missing_files.append(pert_file)


# -------------------------------------------------
# Summary
# -------------------------------------------------
print("\nFinished loading perturbation files.")
print("Alphas loaded:", list(alpha))
print("Sigmas loaded:", list(sigma))
print("k indices:", k_indices)
print("Number of files loaded:", total_loaded_files)

expected_files = len(alpha) * len(sigma) * len(k_indices)
print("Expected number of files:", expected_files)

if missing_files:
    print(
        f"\033[93mMissing {len(missing_files)} perturbation files "
        f"(showing up to 20):\033[0m"
    )
    for filename in missing_files[:20]:
        print("  -", filename)
else:
    print("\033[92mAll perturbation files were found.\033[0m")


## 2. Conformal-time evolution of the Weyl potential — paper Fig. 13

For each perturbation output we construct the Weyl potential
$
\Phi_{\rm W}=\frac{\Phi+\Psi}{2},
$
and evaluate its conformal-time derivative numerically from the sampled perturbation
solution,
$
\Phi_{\rm W}'=\frac{d\Phi_{\rm W}}{d\eta}.
$

The paper shows this ISW source for
$k=0.005,\ 0.05,\ 0.5\ {\rm Mpc}^{-1}$ over the late-time interval $0\leq z\lesssim3$.
Negative values correspond to a decay of the Weyl potential in the sign convention used
in the paper.


In [ ]:

# -------------------------------------------------
# Plot style
# -------------------------------------------------
text_size = 30
fig_size_x = 28
fig_size_y = 10
lw_f = 3

plt.rc('text', usetex=True)
plt.rc('font', family='normal', weight='bold', size=text_size)

# Fixed paper-wide model-to-colour mapping.
colors = [
    '#000000',  # (sigma=0.001, alpha=0.001)
    '#0072B2',  # (sigma=0.3,   alpha=0.1)
    '#E69F00',  # (sigma=0.3,   alpha=0.001)
    '#009E73',  # (sigma=0.3,   alpha=-0.1)
    '#D55E00',  # (sigma=1,     alpha=0.1)
    '#56B4E9',  # (sigma=1,     alpha=-0.1)
    '#CC79A7',  # (sigma=1.5,   alpha=0.1)
    '#F0E442',  # (sigma=1.5,   alpha=0.001)
    '#882255',  # (sigma=1.5,   alpha=-0.05)
]

alpha_consts = [
    "0.001", "0.1", "0.001", "-0.1",
    "0.1", "-0.1", "0.1", "0.001", "-0.05"
]
sigma_consts = [
    "0.001", "0.3", "0.3", "0.3",
    "1.", "1.", "1.5", "1.5", "1.5"
]


# -------------------------------------------------
# Perturbation-output wavenumbers
# -------------------------------------------------
k_output_values_sorted = sorted(k_output_values)
k_value_of_index = dict(zip(k_indices, k_output_values_sorted))
# k_value_of_index = dict(zip(k_indices, k_output_values))

k_labels = {
    k_ind: rf'$k = {k_val:g}\ {{\rm Mpc}}^{{-1}}$'
    for k_ind, k_val in k_value_of_index.items()
}

# Modes shown in paper Fig. 13.
target_k_values = [0.005, 0.05, 0.5]

k_indices_plot = [
    min(k_value_of_index, key=lambda i: abs(k_value_of_index[i] - k_target))
    for k_target in target_k_values
]

print("Plotting k indices:", k_indices_plot)
print("Physical k values:", [k_value_of_index[i] for i in k_indices_plot])


# -------------------------------------------------
# Quantity and time variable
# -------------------------------------------------
quantity_to_plot = "weyl_prime"
x_variable = "z"


# -------------------------------------------------
# Helpers
# -------------------------------------------------
def get_perturbation_time_series(
    pert_arr,
    x_variable="z",
    quantity="weyl_prime"
):
    """
    Extract the metric potentials and their conformal-time derivatives
    from the CLASS scalar perturbation output.

    Relevant columns:
      1: tau  -> Python index 0
      2: a    -> Python index 1
     11: psi  -> Python index 10
     12: phi  -> Python index 11

    A prime denotes differentiation with respect to conformal time eta.
    """

    if not isinstance(pert_arr, np.ndarray):
        raise TypeError(
            f"pert_arr is not a numpy array, got {type(pert_arr)}"
        )

    tau = pert_arr[:, 0]
    a = pert_arr[:, 1]
    psi = pert_arr[:, 10]
    phi = pert_arr[:, 11]

    valid = (
        np.isfinite(tau)
        & np.isfinite(a)
        & np.isfinite(phi)
        & np.isfinite(psi)
        & (a > 0)
    )

    tau = tau[valid]
    a = a[valid]
    phi = phi[valid]
    psi = psi[valid]

    # Evaluate derivatives on data ordered in conformal time.
    sort_tau = np.argsort(tau)
    tau = tau[sort_tau]
    a = a[sort_tau]
    phi = phi[sort_tau]
    psi = psi[sort_tau]

    weyl = 0.5 * (phi + psi)

    # Numerical conformal-time derivatives of the transfer functions.
    phi_prime = np.gradient(phi, tau)
    psi_prime = np.gradient(psi, tau)
    weyl_prime = np.gradient(weyl, tau)

    if quantity == "phi":
        y = phi
    elif quantity == "psi":
        y = psi
    elif quantity == "weyl":
        y = weyl
    elif quantity == "phi_prime":
        y = phi_prime
    elif quantity == "psi_prime":
        y = psi_prime
    elif quantity == "weyl_prime":
        y = weyl_prime
    else:
        raise ValueError(
            "quantity must be 'phi', 'psi', 'weyl', "
            "'phi_prime', 'psi_prime', or 'weyl_prime'."
        )

    if x_variable == "a":
        x = a
    elif x_variable == "z":
        x = 1.0 / a - 1.0
    elif x_variable == "1+z":
        x = 1.0 / a
    elif x_variable == "tau":
        x = tau
    else:
        raise ValueError(
            "x_variable must be 'a', 'z', '1+z', or 'tau'."
        )

    # Sorting here changes only the plotting order.
    sort_x = np.argsort(x)
    return x[sort_x], y[sort_x]


def ylabel_for_quantity(quantity):
    if quantity == "phi":
        return r'$\Phi$'
    elif quantity == "psi":
        return r'$\Psi$'
    elif quantity == "weyl":
        return r'$\Phi_{\rm W}=(\Phi+\Psi)/2$'
    elif quantity == "phi_prime":
        return r"$\Phi'$"
    elif quantity == "psi_prime":
        return r"$\Psi'$"
    elif quantity == "weyl_prime":
        return r"$\Phi_{\rm W}'\ [{\rm Mpc}^{-1}]$"
    return r'$X$'


# -------------------------------------------------
# Check data container
# -------------------------------------------------
if "perturbations" not in data:
    raise KeyError(
        "data['perturbations'] not found. Load the perturbation files first."
    )


# -------------------------------------------------
# Figure: three Fourier modes
# -------------------------------------------------
fig = plt.figure(
    figsize=(fig_size_x, fig_size_y),
    facecolor='w'
)

gs = fig.add_gridspec(
    1, 3,
    wspace=0.17
)

ax_k0 = fig.add_subplot(gs[0, 0])
ax_k1 = fig.add_subplot(gs[0, 1])
ax_k2 = fig.add_subplot(gs[0, 2])

axes = [ax_k0, ax_k1, ax_k2]


# -------------------------------------------------
# Axis style
# -------------------------------------------------
for ax in axes:
    ax.tick_params(
        which='both',
        direction='in',
        top=True,
        right=True
    )
    ax.grid(True, which='both', alpha=0.4)
    ax.minorticks_on()
    ax.yaxis.set_major_locator(MaxNLocator(nbins=6))
    ax.yaxis.set_minor_locator(AutoMinorLocator())


# -------------------------------------------------
# Plot models
# -------------------------------------------------
n_plotted = 0

for ax, k_ind in zip(axes, k_indices_plot):

    for num, (sigma_l, alpha_l) in enumerate(
        zip(sigma_consts, alpha_consts)
    ):

        sig_key = skey(sigma_l)
        alp_key = akey(alpha_l)

        if sig_key not in data["perturbations"]:
            print(
                f"Skipping {sig_key}, {alp_key}, k{k_ind}: "
                "sigma key not found."
            )
            continue

        if alp_key not in data["perturbations"][sig_key]:
            print(
                f"Skipping {sig_key}, {alp_key}, k{k_ind}: "
                "alpha key not found."
            )
            continue

        if k_ind not in data["perturbations"][sig_key][alp_key]:
            print(
                f"Skipping {sig_key}, {alp_key}, k{k_ind}: "
                "k index not found."
            )
            continue

        pert_arr = data["perturbations"][sig_key][alp_key][k_ind]

        if not isinstance(pert_arr, np.ndarray):
            print(
                f"Skipping {sig_key}, {alp_key}, k{k_ind}: "
                "perturbation file not loaded."
            )
            continue

        x, y = get_perturbation_time_series(
            pert_arr,
            x_variable=x_variable,
            quantity=quantity_to_plot
        )

        if len(x) == 0:
            print(
                f"Skipping {sig_key}, {alp_key}, k{k_ind}: "
                "empty valid array."
            )
            continue

        color = colors[num]

        sigma_val = float(sigma_l)
        alpha_val = float(alpha_l)
        label = rf'$(\sigma={sigma_val:g},\,\alpha={alpha_val:g})$'

        ax.plot(
            x,
            y,
            '-',
            lw=lw_f,
            c=color,
            label=label
        )

        n_plotted += 1

    ax.text(
        0.02,
        0.98,
        k_labels[k_ind],
        transform=ax.transAxes,
        fontsize=30,
        verticalalignment='top'
    )

print("Number of curves plotted:", n_plotted)


# -------------------------------------------------
# Labels and ranges
# -------------------------------------------------
axes[0].set_ylabel(
    ylabel_for_quantity(quantity_to_plot),
    fontsize=34
)

for ax in axes:
    ax.set_xlabel(r'$z$', fontsize=34)
    ax.set_xlim(-0.05, 3.3)

ax_k0.set_ylim(-1.e-4, 0)
ax_k1.set_ylim(-1.e-5, 0)
ax_k2.set_ylim(-5.e-7, 0)


# -------------------------------------------------
# Legend
# -------------------------------------------------
axes[0].legend(
    loc='upper left',
    frameon=True,
    framealpha=0.4,
    bbox_to_anchor=(0.2, 0.64),
    fontsize=25
)


# -------------------------------------------------
# Save and show
# -------------------------------------------------
# plt.tight_layout()

os.makedirs("./Figs", exist_ok=True)

plt.savefig(
    './Figs/Weyl_prime_time_evolution.pdf',
    format='pdf',
    dpi=300,
    bbox_inches='tight',
    pad_inches=0.1
)

plt.show()
